# Laboratorio — Robot de entregas en un almacén

A partir de la **imagen**, construye el MDP y resuélvelo con **Value Iteration** y **Policy Iteration**.

![Mundo del ejercicio](https://drive.google.com/uc?export=view&id=1_sJaD57gHuiz1joEgl4B-u0aDy8jtMDo)



## Convención y notación

$$
s=(row,col)
$$

$$
T(s,a,s')=P(s'\mid s,a)
$$

$$
R(s)
$$

Para Value Iteration:

$$
V_{k+1}(s)
=
R(s)
+
\gamma
\max_a
\sum_{s'}T(s,a,s')V_k(s')
$$

Para Policy Evaluation:

$$
V_{k+1}^{\pi}(s)
=
R(s)
+
\gamma
\sum_{s'}T(s,\pi(s),s')V_k^\pi(s')
$$

### Acciones

```python
UP    = (-1, 0)
DOWN  = ( 1, 0)
LEFT  = ( 0,-1)
RIGHT = ( 0, 1)
```



## Reglas del mundo

El grid tiene **5 filas × 6 columnas**.

### Estados especiales

A partir de la imagen identifica:

- `START`
- estanterías / paredes;
- zona de entrega `+10` (**terminal**);
- estación de carga `+2` (**terminal**);
- peligro mortal `-10` (**terminal**);
- peligros `-3` (**no terminales**);
- celdas de piso resbaloso.

### Recompensa

Usamos la convención del notebook de clase, es decir, **\(R(s)\)**:

- entrega: `+10`;
- carga: `+2`;
- peligro mortal: `-10`;
- peligro: `-3`;
- cualquier otro estado transitable: `-1` (costo por paso).

### Dinámica

La transición depende del **estado actual**:

**Piso normal**

$$
P(\text{dirección elegida})=0.90
$$

$$
P(\text{desviación izquierda})=0.05
$$

$$
P(\text{desviación derecha})=0.05
$$

**Piso resbaloso**

$$
P(\text{dirección elegida})=0.60
$$

$$
P(\text{desviación izquierda})=0.20
$$

$$
P(\text{desviación derecha})=0.20
$$

Si el movimiento sale del grid o golpea una estantería, el robot **permanece en el mismo estado**.

Usa:

$$
\gamma=0.9,\qquad \theta=10^{-4}
$$



## Parte 1 — Modela el MDP

Completa la clase `WarehouseMDP`.

La parte importante no es escribir muchas líneas de código: es traducir correctamente la imagen a:

- estados;
- acciones;
- recompensas;
- terminales;
- obstáculos;
- tipos de piso;
- función de transición.


In [ ]:
import numpy as np

class WarehouseMDP:
    def __init__(self):
        self.height = 5
        self.width = 6

        # Coordenadas leídas de la imagen: (fila, columna).
        self.start = (0, 0)
        self.walls = {(0, 3), (1, 1), (2, 4), (4, 2)}
        self.slippery_states = {(1, 2), (2, 1), (3, 3)}

        self.terminal_states = {
            (0, 5): 10.0,   # entrega
            (2, 2): 2.0,    # estación de carga
            (3, 5): -10.0,  # peligro mortal
        }

        self.danger_states = {
            (1, 4): -3.0,
            (4, 1): -3.0,
        }

        self.living_reward = -1.0
        self.gamma = 0.9

        self.actions = [
            (-1, 0),  # UP
            ( 1, 0),  # DOWN
            ( 0,-1),  # LEFT
            ( 0, 1),  # RIGHT
        ]

    def is_valid_state(self, state):
        """Indica si la celda está dentro del grid y no es una estantería."""
        row, col = state
        return 0 <= row < self.height and 0 <= col < self.width and state not in self.walls

    def states(self):
        """Estados transitables, en orden fila-columna para resultados reproducibles."""
        return [
            (row, col)
            for row in range(self.height)
            for col in range(self.width)
            if self.is_valid_state((row, col))
        ]

    def is_terminal(self, state):
        return state in self.terminal_states

    def get_reward(self, state):
        """Implementa R(s): la recompensa depende del estado que se ocupa."""
        if state in self.terminal_states:
            return self.terminal_states[state]
        if state in self.danger_states:
            return self.danger_states[state]
        return self.living_reward

    def get_transition_probs(self, state, action):
        """
        Devuelve [(next_state, probability), ...]. Las colisiones con un borde
        o una pared se acumulan como probabilidad de permanecer en ``state``.
        """
        if self.is_terminal(state):
            return [(state, 1.0)]

        # Girar a la izquierda/derecha la dirección elegida.
        drow, dcol = action
        left = (-dcol, drow)
        right = (dcol, -drow)

        if state in self.slippery_states:
            probabilities = (0.60, 0.20, 0.20)
        else:
            probabilities = (0.90, 0.05, 0.05)

        transition_probs = {}
        for direction, probability in zip((action, left, right), probabilities):
            next_state = (state[0] + direction[0], state[1] + direction[1])
            if not self.is_valid_state(next_state):
                next_state = state
            transition_probs[next_state] = transition_probs.get(next_state, 0.0) + probability

        return list(transition_probs.items())



### Validación mínima del modelo

Antes de implementar Bellman, valida primero el MDP.


In [ ]:
grid = WarehouseMDP()

S = grid.states()
print("Número de estados:", len(S))

# Cada distribución T(s,a,·) debe sumar 1.
for s in S:
    for a in grid.actions:
        transitions = grid.get_transition_probs(s, a)
        total = sum(p for _, p in transitions)
        assert abs(total - 1.0) < 1e-12

print("✓ Todas las distribuciones de transición suman 1.")



## Parte 2 — Value Iteration

Implementa:

$$
V_{k+1}(s)
=
R(s)+\gamma\max_a
\sum_{s'}T(s,a,s')V_k(s')
$$


In [ ]:
def expected_next_value(grid, state, action, V):
    """Calcula sum_s' T(s, a, s') V(s')."""
    return sum(
        probability * V[next_state]
        for next_state, probability in grid.get_transition_probs(state, action)
    )


def value_iteration(grid, threshold=1e-4, max_iter=10_000):
    """Resuelve el MDP mediante actualizaciones síncronas de Bellman óptimo."""
    V = {state: 0.0 for state in grid.states()}
    for state in grid.terminal_states:
        V[state] = grid.get_reward(state)

    for iteration in range(1, max_iter + 1):
        new_V = V.copy()
        delta = 0.0

        for state in grid.states():
            if grid.is_terminal(state):
                new_V[state] = grid.get_reward(state)
                continue

            best_next_value = max(
                expected_next_value(grid, state, action, V)
                for action in grid.actions
            )
            new_V[state] = grid.get_reward(state) + grid.gamma * best_next_value
            delta = max(delta, abs(new_V[state] - V[state]))

        V = new_V
        if delta < threshold:
            return V, iteration

    raise RuntimeError("Value Iteration no convergió antes de max_iter.")


def extract_policy(grid, V):
    """Extrae pi*(s) usando V; en empates conserva el orden de grid.actions."""
    policy = {}
    for state in grid.states():
        if not grid.is_terminal(state):
            policy[state] = max(
                grid.actions,
                key=lambda action: expected_next_value(grid, state, action, V),
            )
    return policy



## Parte 3 — Policy Iteration

### Policy Evaluation

$$
V_{k+1}^{\pi}(s)
=
R(s)+\gamma
\sum_{s'}T(s,\pi(s),s')V_k^\pi(s')
$$

### Policy Improvement

$$
\pi_{\mathrm{new}}(s)
=
\arg\max_a
\sum_{s'}T(s,a,s')V^\pi(s')
$$

In [ ]:
def policy_evaluation(grid, policy, threshold=1e-4, max_iter=10_000):
    """Evalúa una política fija con actualizaciones síncronas."""
    V = {state: 0.0 for state in grid.states()}
    for state in grid.terminal_states:
        V[state] = grid.get_reward(state)

    for _ in range(max_iter):
        new_V = V.copy()
        delta = 0.0

        for state in grid.states():
            if grid.is_terminal(state):
                new_V[state] = grid.get_reward(state)
                continue

            action = policy[state]
            next_value = expected_next_value(grid, state, action, V)
            new_V[state] = grid.get_reward(state) + grid.gamma * next_value
            delta = max(delta, abs(new_V[state] - V[state]))

        V = new_V
        if delta < threshold:
            return V

    raise RuntimeError("Policy Evaluation no convergió antes de max_iter.")


def policy_improvement(grid, V):
    """Construye la política codiciosa respecto a V."""
    return extract_policy(grid, V)


def policy_iteration(grid, threshold=1e-4, max_iter=100):
    """Alterna evaluación y mejora hasta que la política sea estable."""
    policy = {
        state: grid.actions[0]
        for state in grid.states()
        if not grid.is_terminal(state)
    }
    history = []

    for iteration in range(1, max_iter + 1):
        V = policy_evaluation(grid, policy, threshold=threshold)
        new_policy = policy_improvement(grid, V)
        changed_states = sum(
            policy[state] != new_policy[state]
            for state in policy
        )
        history.append({
            "iteration": iteration,
            "changed_states": changed_states,
            "stable": changed_states == 0,
        })

        policy = new_policy
        if changed_states == 0:
            return policy, V, history

    raise RuntimeError("Policy Iteration no convergió antes de max_iter.")



## Parte 4 — Visualización y comparación


In [ ]:
ARROWS = {
    (-1, 0): "↑",
    ( 1, 0): "↓",
    ( 0,-1): "←",
    ( 0, 1): "→",
}

def print_values(grid, V):
    for r in range(grid.height):
        row = []
        for c in range(grid.width):
            s = (r, c)
            if s in grid.walls:
                row.append("  WALL  ")
            else:
                row.append(f"{V[s]:+7.3f}")
        print(" | ".join(row))


def print_policy(grid, policy):
    for r in range(grid.height):
        row = []
        for c in range(grid.width):
            s = (r, c)

            if s in grid.walls:
                row.append(" # ")
            elif grid.is_terminal(s):
                reward = grid.get_reward(s)
                row.append(f"{reward:+.0f}")
            else:
                row.append(f" {ARROWS[policy[s]]} ")

        print(" | ".join(row))


In [ ]:
# VALUE ITERATION
V_vi, n_vi = value_iteration(grid)
pi_vi = extract_policy(grid, V_vi)

print("=== VALUE ITERATION ===")
print("Iteraciones:", n_vi)
print("\nValores:")
print_values(grid, V_vi)
print("\nPolítica:")
print_policy(grid, pi_vi)


# POLICY ITERATION
pi_pi, V_pi, history = policy_iteration(grid)

print("\n=== POLICY ITERATION ===")
print("Historia:", history)
print("\nValores:")
print_values(grid, V_pi)
print("\nPolítica:")
print_policy(grid, pi_pi)

assert pi_vi == pi_pi
print("\n✓ Ambos algoritmos encontraron la misma política óptima.")



## Parte 5 — Interpreta la política

Antes de cambiar parámetros, responde:

1. Desde `START`, ¿el robot busca la **entrega +10** o prefiere la **estación de carga +2**?
2. ¿Por qué una recompensa menor podría ser óptima?
3. ¿En qué estados el piso resbaloso cambia la decisión?
4. ¿Qué papel cumple el costo por paso `-1`?
5. ¿Por qué \(T(s,a,s')\) ya no puede implementarse con las mismas probabilidades para todos los estados?

### Experimento A — Menos costo por paso

Cambia:

```python
living_reward = -0.1
```

Predice la política **antes de ejecutar**.

### Experimento B — Piso muy resbaloso

Cambia la probabilidad de movimiento deseado del piso resbaloso:

```python
0.60 → 0.40
```

y reparte el restante entre las dos desviaciones.

### Experimento C — Más paciencia

Cambia:

```python
gamma = 0.99
```

¿La política valora más la recompensa `+10` distante?

### Bonus

Encuentra aproximadamente el valor de `living_reward` a partir del cual la política desde `START` cambia entre:

- ir a carga `+2`;
- intentar llegar a entrega `+10`.


### Respuestas

1. Desde START, la política óptima prefiere la estación de carga +2: va a la derecha hasta (0, 2) y luego baja hacia (1, 2) y (2, 2).
2. Una recompensa terminal menor puede ser óptima si está mucho más cerca o si el trayecto hacia la recompensa grande concentra riesgo de penalizaciones. En este caso, la entrega exige pasar cerca del peligro -3; con descuento y costo por paso, importa el retorno esperado completo, no solo el premio final.
3. En una comparación con el piso resbaloso convertido en piso normal, cambia directamente la acción de (1, 2): pasa de bajar hacia carga a ir a la derecha. El cambio de los valores también modifica la decisión de (3, 1), de derecha a arriba. Las celdas resbalosas (2, 1) y (3, 3) se mantienen en la política óptima, pero sus probabilidades de desvío siguen afectando los valores esperados.
4. El costo -1 incentiva trayectorias cortas y evita que el robot dé vueltas o retrase la llegada a un terminal.
5. Porque el tipo de piso forma parte de la dinámica local: desde un estado normal la acción elegida ocurre con probabilidad 0.90, mientras que desde uno resbaloso ocurre con 0.60; por tanto, las probabilidades de transición dependen de s.

Experimento A predicción y resultado. Con living_reward = -0.1, moverse resulta menos costoso. Se espera que la política esté más dispuesta a intentar la entrega +10; al ejecutar, en (1, 2) cambia de bajar hacia carga a ir a la derecha, en dirección a la entrega.

Experimento B — predicción. Si en piso resbaloso la dirección deseada tiene probabilidad 0.40 y cada desvío 0.30, las acciones que atraviesan dichas celdas se vuelven menos atractivas; la política tenderá a rodearlas cuando exista una alternativa segura.

Experimento C — predicción y resultado. Con gamma = 0.99, las recompensas futuras se descuentan menos, así que la entrega distante +10 gana importancia relativa frente a una carga cercana +2. Al ejecutar, también cambia la acción de (1, 2) de bajar a ir a la derecha.

Bonus. Al barrer living_reward, el cambio entre bajar a carga e ir a la derecha hacia entrega en (1, 2) ocurre aproximadamente en -0.80 (entre -0.80 y -0.79 con un barrido de 0.01). La acción inicial de START no revela el cambio porque ambos planes comienzan moviéndose a la derecha.
